 Installer et importer

In [0]:
%pip install tensorflow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 281.9/281.9 MB 110.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 101.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 22.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 86.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 112.6 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.4
    Not uninstalling protobuf at /databricks/python3/lib/python3.12/site-packages, outside environment /local_disk0/.ephemeral_nfs/envs/pythonEnv-29392805-c6f6-481e-8128-efd121c2e8fa
    Can't uninstall 'protobuf'. No files were found to uninstall.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-api-core 2.20.0 requires protobuf!=3.20.0,!=3.20.1,!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.

In [0]:
dbutils.library.restartPython()

In [0]:
import tensorflow as tf
import numpy as np
import pandas as pd
from pyspark.sql.functions import col

print(f" TensorFlow version : {tf.__version__}")

# Charger les données
VOLUME = "/Volumes/workspace/default/ecommerce_data"

ratings_indexed = spark.read.parquet(f"{VOLUME}/ratings_indexed")
print(f" Données chargées : {ratings_indexed.count():,} interactions")

 TensorFlow version : 2.21.0
 Données chargées : 701,528 interactions


 Préparer les données pour TensorFlow

In [0]:
# Convertir en Pandas (TensorFlow n'utilise pas Spark)
# Prendre un sample pour rester rapide
sample = ratings_indexed \
    .select("user_idx", "item_idx", "rating") \
    .sample(fraction=0.1, seed=42)  # 10% des données

df = sample.toPandas()
df["rating"] = df["rating"].astype(float)

print(f" Sample : {len(df):,} interactions")
print(f"Users uniques : {df['user_idx'].nunique():,}")
print(f"Items uniques : {df['item_idx'].nunique():,}")

# Stats
num_users = int(ratings_indexed.select("user_idx").distinct().count())
num_items = int(ratings_indexed.select("item_idx").distinct().count())

print(f"\nTotal users : {num_users:,}")
print(f"Total items : {num_items:,}")

 Sample : 70,005 interactions
Users uniques : 68,680
Items uniques : 32,009

Total users : 631,986
Total items : 112,565


 Split train/test

In [0]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42
)

print(f"Train : {len(train_df):,}")
print(f"Test  : {len(test_df):,}")

# Préparer les arrays numpy
X_train = [
    train_df["user_idx"].values,
    train_df["item_idx"].values
]
y_train = train_df["rating"].values

X_test = [
    test_df["user_idx"].values,
    test_df["item_idx"].values
]
y_test = test_df["rating"].values

Train : 56,004
Test  : 14,001


Construire le modèle

In [0]:
EMBEDDING_DIM = 32  # dimensions des embeddings

# ── Input layers ──────────────────────────
user_input = tf.keras.Input(shape=(1,), name="user_input")
item_input = tf.keras.Input(shape=(1,), name="item_input")

# ── Embedding layers ───────────────────────
user_embedding = tf.keras.layers.Embedding(
    input_dim=num_users + 1,
    output_dim=EMBEDDING_DIM,
    name="user_embedding"
)(user_input)

item_embedding = tf.keras.layers.Embedding(
    input_dim=num_items + 1,
    output_dim=EMBEDDING_DIM,
    name="item_embedding"
)(item_input)

# ── Flatten ────────────────────────────────
user_vec = tf.keras.layers.Flatten()(user_embedding)
item_vec = tf.keras.layers.Flatten()(item_embedding)

# ── Concatener et Dense layers ─────────────
concat = tf.keras.layers.Concatenate()([user_vec, item_vec])
dense1 = tf.keras.layers.Dense(64, activation="relu")(concat)
dense2 = tf.keras.layers.Dense(32, activation="relu")(dense1)
output = tf.keras.layers.Dense(1)(dense2)

# ── Compiler ───────────────────────────────
model_tf = tf.keras.Model(
    inputs=[user_input, item_input],
    outputs=output
)

model_tf.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

model_tf.summary()
print(" Modèle TensorFlow créé !")

W0000 00:00:1773093952.979088    3675 cpu_allocator_impl.cc:82] Allocation of 80894336 exceeds 10% of free system memory.
W0000 00:00:1773093953.042890    3675 cpu_allocator_impl.cc:82] Allocation of 80894336 exceeds 10% of free system memory.
W0000 00:00:1773093953.059060    3675 cpu_allocator_impl.cc:82] Allocation of 80894336 exceeds 10% of free system memory.
W0000 00:00:1773093953.113115    3675 cpu_allocator_impl.cc:82] Allocation of 14408448 exceeds 10% of free system memory.
W0000 00:00:1773093953.130738    3675 cpu_allocator_impl.cc:82] Allocation of 14408448 exceeds 10% of free system memory.


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ user_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_input          │ (None, 1)         │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ user_embedding      │ (None, 1, 32)     │ 20,223,584 │ user_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ item_embedding      │ (None, 1, 32)     │  3,602,112 │ item_input[0][0]  │
│ (Embedding)         │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten (Flatten)   │ (None, 32)        │          0 │ user_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ flatten_1 (Flatten) │ (None, 32)        │          0 │ item_embedding[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 64)        │          0 │ flatten[0][0],    │
│ (Concatenate)       │                   │            │ flatten_1[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 64)        │      4,160 │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 32)        │      2,080 │ dense[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_2 (Dense)     │ (None, 1)         │         33 │ dense_1[0][0]     │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 23,831,969 (90.91 MB)

 Trainable params: 23,831,969 (90.91 MB)

 Non-trainable params: 0 (0.00 B)

 Modèle TensorFlow créé !


 Entraîner
 

In [0]:
print(" Entraînement TensorFlow...")

history = model_tf.fit(
    X_train,
    y_train,
    epochs=5,           # 5 epochs = rapide
    batch_size=512,
    validation_split=0.1,
    verbose=1
)

print("Entraînement terminé !")

 Entraînement TensorFlow...
Epoch 1/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 12s 111ms/step - loss: 9.0990 - mae: 2.5795 - val_loss: 2.3090 - val_mae: 1.2514
Epoch 2/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 11s 111ms/step - loss: 1.7250 - mae: 1.0573 - val_loss: 2.3882 - val_mae: 1.3321
Epoch 3/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 11s 114ms/step - loss: 0.4576 - mae: 0.5000 - val_loss: 2.7112 - val_mae: 1.4562
Epoch 4/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 11s 115ms/step - loss: 0.2044 - mae: 0.3540 - val_loss: 2.7273 - val_mae: 1.4599
Epoch 5/5
99/99 ━━━━━━━━━━━━━━━━━━━━ 21s 122ms/step - loss: 0.1575 - mae: 0.3179 - val_loss: 2.6254 - val_mae: 1.4328
Entraînement terminé !


Évaluer et comparer avec ALS

In [0]:
from sklearn.metrics import mean_squared_error
import numpy as np

# Correction : model_tf au lieu de model
rmse_als = 2.5304
predictions_tf = model_tf.predict(X_test).flatten()
rmse_tf = np.sqrt(mean_squared_error(y_test, predictions_tf))

print("=" * 45)
print(" COMPARAISON ALS vs TensorFlow")
print("=" * 45)
print(f"RMSE ALS        = {rmse_als:.4f}")
print(f"RMSE TensorFlow = {rmse_tf:.4f}")
print(f"Amélioration    = -{rmse_als - rmse_tf:.4f}")
print(f"                = {(rmse_als - rmse_tf)/rmse_als*100:.1f}% meilleur")

print("\nRÉSUMÉ POUR LA SOUTENANCE")
print("=" * 45)
print(f"TensorFlow bat ALS de {rmse_als - rmse_tf:.4f} points RMSE")
print(f"Gain de {(rmse_als - rmse_tf)/rmse_als*100:.1f}%")
print(f"\nALS = rapide et scalable (457K users, 80K items)")
print(f"TF  = plus précis (RMSE {rmse_tf:.4f} vs {rmse_als:.4f})")

438/438 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
 COMPARAISON ALS vs TensorFlow
RMSE ALS        = 2.5304
RMSE TensorFlow = 1.6110
Amélioration    = -0.9194
                = 36.3% meilleur

RÉSUMÉ POUR LA SOUTENANCE
TensorFlow bat ALS de 0.9194 points RMSE
Gain de 36.3%

ALS = rapide et scalable (457K users, 80K items)
TF  = plus précis (RMSE 1.6110 vs 2.5304)


Sauvegarder les resultats

In [0]:
import json

resultats_finaux = {
    "rmse_als"        : 2.5304,
    "rmse_tensorflow" : float(rmse_tf),
    "amelioration_pct": float((rmse_als - rmse_tf) / rmse_als * 100),
    "embedding_dim"   : 32,
    "epochs"          : 5
}

with open(f"{VOLUME}/comparaison_modeles.json", "w") as f:
    json.dump(resultats_finaux, f, indent=2)

print("Résultats sauvegardés !")
print(json.dumps(resultats_finaux, indent=2))



Résultats sauvegardés !
{
  "rmse_als": 2.5304,
  "rmse_tensorflow": 1.6110073922274468,
  "amelioration_pct": 36.33388427808067,
  "embedding_dim": 32,
  "epochs": 5
}
